In [2]:
import pandas as pd
import os 
import numpy as np
import geopandas as gpd

1. Load all separate datasets and combine into one csv

- This saves out a csv with all warnings together in one file: /home/csutter/DRIVE-clean/snow_squall/data/nws_warnings/nws_all_warnings.csv
- Don't run this again

In [ ]:
dir = "/home/csutter/DRIVE-clean/snow_squall/data/nws_warnings"
files = os.listdir(dir)

xs = [f"{dir}/{f}" for f in files]

print(len(xs))
print(xs[0:3])

In [ ]:
ds = []

for x in xs:
    d = pd.read_excel(x)
    ds.append(d)

df = pd.concat(ds)

In [ ]:
df.head(4)

In [ ]:
df.columns

In [ ]:
# df.to_csv("/home/csutter/DRIVE-clean/snow_squall/data/nws_warnings/nws_all_warnings.csv")

2. Add in geometry for NWS regions
- Don't run this again. Keeping for reference for mapping to zones and counties geometries
- This uses the combined csv made from step 1 to add in spatial information

In [3]:
# Load data from step 1

# From NWS data: https://mesonet.agron.iastate.edu/vtec/search.php?mode=byugc&state=NY
warnings_df = pd.read_csv("/home/csutter/DRIVE-clean/snow_squall/data/nws_warnings/nws_all_warnings.csv")
# the column in here, 'ugc' is universal geographic code -- these include county and zone locations. County codes look like "NYC003" and zone codes look like "NYZ101".
# From iastate data site above: The NWS issues watch, warnings, and advisories (WWA) for counties/parishes. For some products (like winter warnings), they issue for forecast zones. In many parts of the country, these zones are exactly the same as the counties/parishes. When you get into regions with topography, then zones will start to differ to the local counties.
# for ease of use, add a col indicating whether warning is Zone or County
warnings_df["location_type"] = warnings_df['ugc'].apply(lambda x: "zone" if "NYZ" in x else "county")

# Load GIS data
# https://www.weather.gov/gis/IDP-GISRestMetadata
# Load zones dataset https://www.weather.gov/gis/publiczones
zones_gdf = gpd.read_file('/home/csutter/DRIVE-clean/snow_squall/data/shapefile/z_18mr25/z_18mr25.shp')
# There are specific fire zones and marine zones, which are in different shapefiles. 
firezones_gdf = gpd.read_file('/home/csutter/DRIVE-clean/snow_squall/data/shapefile/fz18mr25') # https://www.weather.gov/gis/firezones. # this shapefle critical b/c NWS warnings have zones that are new firezones (https://www.weather.gov/okx/FireWeatherZoneChanges), which aren't included/up-to-date in the main zones_gdf above
marinezones_gdf = gpd.read_file('/home/csutter/DRIVE-clean/snow_squall/data/shapefile/mz18mr25') # https://www.weather.gov/gis/MarineZones
# Load counties https://www.weather.gov/gis/Counties 
counties_gdf = gpd.read_file('/home/csutter/DRIVE-clean/snow_squall/data/shapefile/c_18mr25') 
# the column in here is "FIPS" (Federal Information Processing Standard) is the federal county code, it maps to the 6 digit NY county code (e.g. NYC003) in the NWS warning dataset by based on "NYC" followed by the the last 3 digits of FIPS Id. Source for that mapping is on page 5 here: https://www.weather.gov/media/directives/010_pdfs_archived/pd01017002b.pdf 


# Subset the NY locations. This is only critical for the ny_counties one due to the way we create the county code from the FIPS code. However, subsetting the other dfs to work with smaller datasets.
ny_zones = zones_gdf[zones_gdf["STATE"]=="NY"]
ny_firezones = firezones_gdf[firezones_gdf["STATE"]=="NY"] 
ny_marinezones = marinezones_gdf # marinezones doesnt have state designation to filter, but keep similar naming convention
ny_counties = counties_gdf[counties_gdf["STATE"]=="NY"]

# Prepare data - format zone and county codes so that join works with the NWS dataset. See documentation above.
ny_zones["ugc_gis"] = ny_zones['STATE_ZONE'].apply(lambda x: x.replace("NY", "NYZ"))
ny_firezones["ugc_gis"] = ny_firezones['STATE_ZONE'].apply(lambda x: x.replace("NY", "NYZ"))
ny_marinezones["ugc_gis"] = ny_marinezones['ID'].apply(lambda x: x.replace("NY", "NYZ"))
ny_counties["ugc_gis"] = ny_counties['FIPS'].apply(lambda x: f"NYC{x[-3:]}") # last 3 digits
ny_zones["loc_desc"] = ny_zones["NAME"]
ny_counties["loc_desc"] = ny_counties["COUNTYNAME"]
ny_firezones["loc_desc"] = ny_firezones["NAME"]
ny_marinezones["loc_desc"] = ny_marinezones["NAME"]

# Inspect the geometries. Don't want to have multiple CRS (coordinate reference systems) in one dataset
z = ny_zones[["ugc_gis","geometry","loc_desc"]]
fz = ny_firezones[["ugc_gis","geometry","loc_desc"]]
mz = ny_marinezones[["ugc_gis","geometry","loc_desc"]]
c = ny_counties[["ugc_gis","geometry","loc_desc"]]


print("zones_slice.crs:", z.crs, "type:", type(z))
print("counties_slice.crs:", c.crs, "type:", type(c))
# different CRS  - convert the county crs to use the zone crs
c = c.to_crs(z.crs)
mz = mz.to_crs(z.crs)
print("new")
print("counties_slice.crs:", c.crs, "type:", type(c))

# After aligning CRS, can concat the two GIS datasets:
gisdf = pd.concat([z,c,fz,mz])#
print("gisdf type")
print(type(gisdf))

# Join the NWS warnings data with GIS data
warnings = warnings_df.merge(
    gisdf, 
    left_on='ugc', 
    right_on='ugc_gis', 
    how='left'
)

# want to save out the geometry for future use (need to save it as a geopandas df -- so convert the merged df (that has the warning metadata (warnings_df) and the gisdf) to a GeoDataFrame

warnings = gpd.GeoDataFrame(warnings, geometry="geometry", crs=gisdf.crs)
print(type(warnings))

print("Confirm no duplicate entries, lengths of NWS warnings df and the final cleaned df should be the same")
print(len(warnings_df))
print(len(warnings))

zones_slice.crs: GEOGCRS["NAD83",DATUM["North American Datum 1983",ELLIPSOID["GRS 1980",6378137,298.257222101,LENGTHUNIT["metre",1]],ID["EPSG",6269]],PRIMEM["Greenwich",0,ANGLEUNIT["Degree",0.0174532925199433]],CS[ellipsoidal,3],AXIS["longitude",east,ORDER[1],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["latitude",north,ORDER[2],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["ellipsoidal height (h)",up,ORDER[3],LENGTHUNIT["metre",1,ID["EPSG",9001]]]] type: <class 'geopandas.geodataframe.GeoDataFrame'>
counties_slice.crs: epsg:4269 type: <class 'geopandas.geodataframe.GeoDataFrame'>
new
counties_slice.crs: GEOGCRS["NAD83",DATUM["North American Datum 1983",ELLIPSOID["GRS 1980",6378137,298.257222101,LENGTHUNIT["metre",1]],ID["EPSG",6269]],PRIMEM["Greenwich",0,ANGLEUNIT["Degree",0.0174532925199433]],CS[ellipsoidal,3],AXIS["longitude",east,ORDER[1],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["latitude",north,ORDER[2],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["ellipsoidal height (h)",up,OR

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

3. Add duration of event in a new col

In [4]:
warnings['issued'] = pd.to_datetime(warnings['issued'])
warnings['expired'] = pd.to_datetime(warnings['expired'])
warnings["duration"] = warnings["expired"] -warnings["issued"]
warnings["duration_sec"] = warnings["duration"].dt.total_seconds() # geopandas df can't save a timedelta "duration" col so convert it to seconds and drop the timedelta columns
warnings = warnings.drop(columns=["duration"])

Save out final cleaned csv with info added, including: geometry, duration, location type, location desc

In [5]:
type(warnings)

geopandas.geodataframe.GeoDataFrame

In [6]:
# Save out to the geopandas df

# warnings.to_file("/home/csutter/DRIVE-clean/snow_squall/data/nws_warnings/nws_all_warnings_cleaned.gpkg", driver="GPKG")

In [7]:
warnings.dtypes

Unnamed: 0                int64
vtec_year                 int64
iso_issued               object
issued           datetime64[ns]
iso_expired              object
expired          datetime64[ns]
eventid                   int64
phenomena                object
significance             object
hvtec_nwsli              object
wfo                      object
ugc                      object
product_id               object
name                     object
ph_name                  object
sig_name                 object
url                      object
location_type            object
ugc_gis                  object
geometry               geometry
loc_desc                 object
duration_sec            float64
dtype: object

In [8]:
warnings.head(4)

,Unnamed: 0,vtec_year,iso_issued,issued,iso_expired,expired,eventid,phenomena,significance,hvtec_nwsli,...,product_id,name,ph_name,sig_name,url,location_type,ugc_gis,geometry,loc_desc,duration_sec
0,0,2022,2022-01-17T07:39:00Z,2022-01-17 07:39:00,2022-01-17T08:45:00Z,2022-01-17 08:45:00,1,SV,W,NaN,...,202201170739-KOKX-WUUS51-SVROKX,Severe Thunderstorm Warning,Severe Thunderstorm,Warning,/vtec/?year=2022&wfo=KOKX&phenomena=SV&signifi...,county,NYC081,"MULTIPOLYGON (((-73.80570 40.60329, -73.80680 ...",Queens,3960.0
1,1,2022,2022-02-18T11:37:00Z,2022-02-18 11:37:00,2022-02-18T12:11:00Z,2022-02-18 12:11:00,2,SV,W,NaN,...,202202181137-KOKX-WUUS51-SVROKX,Severe Thunderstorm Warning,Severe Thunderstorm,Warning,/vtec/?year=2022&wfo=KOKX&phenomena=SV&signifi...,county,NYC081,"MULTIPOLYGON (((-73.80570 40.60329, -73.80680 ...",Queens,2040.0
2,2,2022,2022-02-19T20:07:00Z,2022-02-19 20:07:00,2022-02-19T20:45:00Z,2022-02-19 20:45:00,5,SQ,W,NaN,...,202202192007-KOKX-WWUS51-SQWOKX,Snow Squall Warning,Snow Squall,Warning,/vtec/?year=2022&wfo=KOKX&phenomena=SQ&signifi...,county,NYC081,"MULTIPOLYGON (((-73.80570 40.60329, -73.80680 ...",Queens,2280.0
3,3,2022,2022-03-07T23:25:00Z,2022-03-07 23:25:00,2022-03-08T02:55:00Z,2022-03-08 02:55:00,38,SV,A,NaN,...,202203072325-KOKX-WWUS61-WCNOKX,Severe Thunderstorm Watch,Severe Thunderstorm,Watch,/vtec/?year=2022&wfo=KOKX&phenomena=SV&signifi...,county,NYC081,"MULTIPOLYGON (((-73.80570 40.60329, -73.80680 ...",Queens,12600.0
